# Greedy — Cheat Sheet

**Companion to:** DP Study Guide, Backtracking Cheat Sheet

**What this adds:** Greedy was not covered in any study guide. Unlike Backtracking (universal template) or DP (5-step framework), Greedy has no single template — the structure varies by problem type. This cheat sheet is organized by problem type rather than by template.

---
## When to Use

| Signal | Pattern to reach for |
| :--- | :--- |
| "minimum number of intervals", "non-overlapping" | Interval scheduling |
| "meeting rooms", "can attend all meetings" | Interval overlap detection |
| "can reach the end", "minimum jumps" | Jump game |
| "minimum cost to connect", "huffman" | Greedy with heap |
| "assign tasks", "match pairs optimally" | Greedy sorting + matching |
| "maximize/minimize with a local choice" | Greedy (if optimal substructure + greedy choice property holds) |

---
## Patterns at a Glance

| Pattern | Greedy choice | Sort by | Problems |
| :--- | :--- | :--- | :--- |
| Interval scheduling | Pick interval ending earliest | End time ASC | 435, 452, 56 |
| Interval overlap detection | Track max end of overlapping group | Start time ASC | 56, 57, 986 |
| Jump game | Maximize reachable index at each step | No sort | 55, 45 |
| Greedy with heap | Always pick the smallest/largest available | Heap order | 621, 767, 1405 |
| Sorting + matching | Sort both arrays, match greedily | Varies | 455, 881, 870 |
| Prefix / running choice | Make locally optimal decision at each position | No sort | 134, 135, 406 |

---
# Part 0 — Quick Recap

A greedy algorithm makes the **locally optimal choice at each step** without backtracking, and this leads to a globally optimal solution. It works when two properties hold:

1. **Greedy choice property** — the local optimum at each step is always part of some global optimum
2. **Optimal substructure** — the optimal solution to the whole problem contains optimal solutions to subproblems

**How it differs from DP:** DP considers all choices and stores results. Greedy commits to one choice immediately and never revisits it. Greedy is faster (usually O(n log n) vs O(n²) or more for DP) but only correct when the greedy choice property holds.

**How it differs from Backtracking:** Backtracking explores all paths and backtracks on dead ends. Greedy never backtracks — it commits to each choice permanently.

**The hardest part of greedy problems:** recognizing that greedy works and proving why. If you're unsure, ask: "Can I construct a counterexample where the local optimum leads to a globally wrong answer?" If yes — use DP. If no — greedy is likely correct.

---
# Part 1 — Interval Scheduling

## Core idea

Given a set of intervals, select the maximum number of non-overlapping intervals, or find the minimum number needed to cover/remove.

**The key greedy insight:** always pick the interval that **ends earliest** — it leaves the most room for future intervals. Sort by end time, then greedily select intervals that don't overlap with the last selected.

**Overlap condition:** two intervals `[a, b]` and `[c, d]` overlap if `c < b` (start of second is before end of first).

In [ ]:
# Template — Maximum non-overlapping intervals (LC 435)
# Returns minimum number of intervals to REMOVE
def eraseOverlapIntervals(intervals):
    intervals.sort(key=lambda x: x[1])  # sort by END time
    removed = 0
    last_end = float('-inf')
    for start, end in intervals:
        if start < last_end:            # overlaps with previous — remove it
            removed += 1
        else:
            last_end = end              # no overlap — keep it, update last_end
    return removed


# Template — Minimum arrows to burst balloons (LC 452)
# Same pattern: sort by end, shoot at last_end
def findMinArrowShots(points):
    points.sort(key=lambda x: x[1])    # sort by END
    arrows = 1
    last_end = points[0][1]
    for start, end in points[1:]:
        if start > last_end:            # balloon not reached by current arrow
            arrows += 1
            last_end = end
    return arrows


# Template — Merge overlapping intervals (LC 56)
# Sort by START time — merge when next start <= current end
def merge(intervals):
    intervals.sort(key=lambda x: x[0])  # sort by START time
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:       # overlaps — extend current interval
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])  # no overlap — new interval
    return merged

## Common mistakes

- Sorting by start instead of end for non-overlapping selection — sorting by start doesn't minimize conflicts; always sort by end for scheduling
- Using `<=` vs `<` for overlap condition — `[1,2]` and `[2,3]` are NOT overlapping (they share a boundary); use `start < last_end` not `<=`
- Forgetting to update `last_end` only when keeping an interval, not when removing

**Practice problems:**

| Problem | Sort by | Greedy choice | Notes |
| :--- | :--- | :--- | :--- |
| LC 435 — Non-overlapping Intervals | End ASC | Keep earliest-ending, remove overlap | Count removals |
| LC 452 — Minimum Arrows | End ASC | Shoot at current end | Count arrows |
| LC 56 — Merge Intervals | Start ASC | Extend if overlapping | Build merged list |

---
# Part 2 — Jump Game

## Core idea

At each position, you can jump up to `nums[i]` steps forward. The greedy insight: track the **maximum reachable index** as you scan left to right. You never need to simulate individual jumps — just track how far you can reach.

**For minimum jumps (LC 45):** track the current jump boundary and the farthest reachable within that boundary. When you reach the boundary, you must take another jump.

In [ ]:
# LC 55 — Jump Game (can you reach the end?)
def canJump(nums):
    max_reach = 0
    for i, jump in enumerate(nums):
        if i > max_reach: return False  # current index is unreachable
        max_reach = max(max_reach, i + jump)
    return True


# LC 45 — Jump Game II (minimum jumps to reach end)
def jump(nums):
    jumps = 0
    current_end = 0                     # boundary of current jump range
    farthest = 0                        # farthest reachable within current range
    for i in range(len(nums) - 1):     # don't need to jump from last position
        farthest = max(farthest, i + nums[i])
        if i == current_end:            # reached boundary — must jump
            jumps += 1
            current_end = farthest
    return jumps

## Common mistakes

- Simulating individual jumps instead of tracking max reachable — O(n²) vs O(n)
- In LC 45: iterating to `len(nums)` instead of `len(nums) - 1` — causes an off-by-one extra jump
- Confusing `current_end` (boundary of current jump) with `farthest` (best reachable from current jump)

**Practice problems:**

| Problem | Notes |
| :--- | :--- |
| LC 55 — Jump Game | Can reach? Track max_reach, return False if i > max_reach |
| LC 45 — Jump Game II | Min jumps: boundary + farthest pattern |
| LC 1696 — Jump Game VI | Max score: monotonic deque DP hybrid |

---
# Part 3 — Greedy with Heap

## Core idea

When the greedy choice at each step is "pick the most/least frequent" or "pick the largest/smallest available", use a max-heap (negate for Python's min-heap) to always access the optimal choice in O(log n).

In [ ]:
import heapq
from collections import Counter

# LC 621 — Task Scheduler
# Minimum time to finish all tasks with cooldown n between same tasks
# Greedy: always schedule the most frequent remaining task
def leastInterval(tasks, n):
    freq = Counter(tasks)
    max_heap = [-f for f in freq.values()]  # max-heap (negate)
    heapq.heapify(max_heap)
    time = 0
    queue = []                              # (next_available_time, freq)

    while max_heap or queue:
        time += 1
        if max_heap:
            freq = heapq.heappop(max_heap) + 1  # use one instance (negate back)
            if freq < 0:                         # still has remaining count
                queue.append((time + n, freq))   # available again after cooldown
        if queue and queue[0][0] == time:        # task cooled down
            heapq.heappush(max_heap, queue.pop(0)[1])
    return time


# LC 767 — Reorganize String
# Rearrange so no two adjacent chars are same
# Greedy: always place the most frequent remaining character
def reorganizeString(s):
    freq = Counter(s)
    max_heap = [(-f, ch) for ch, f in freq.items()]
    heapq.heapify(max_heap)
    res = []
    prev_f, prev_ch = 0, ''

    while max_heap:
        f, ch = heapq.heappop(max_heap)
        res.append(ch)
        if prev_f < 0:                      # previous char still has remaining count
            heapq.heappush(max_heap, (prev_f, prev_ch))
        prev_f, prev_ch = f + 1, ch         # decrement count (negated)

    return ''.join(res) if len(res) == len(s) else ''

## Common mistakes

- Forgetting Python only has min-heap — negate values to simulate max-heap
- In task scheduler: using idle time formula shortcut without understanding it — the heap simulation is more generalizable
- Not re-adding the previous character to the heap before placing a new one — causes adjacent duplicates

**Practice problems:**

| Problem | Heap type | Notes |
| :--- | :--- | :--- |
| LC 621 — Task Scheduler | Max-heap on frequency | Cooldown queue + heap |
| LC 767 — Reorganize String | Max-heap on frequency | Place most frequent, hold previous |
| LC 1405 — Longest Happy String | Max-heap on frequency | Same as 767 but max 2 consecutive |

---
# Part 4 — Sorting + Matching

## Core idea

Sort one or both arrays, then match greedily. The sort order depends on what you're optimizing — match smallest to smallest, or smallest need to smallest supply.

**Key insight:** after sorting, you never need to consider non-adjacent pairs. The optimal match for the current element is always the closest valid element in the sorted other array.

In [ ]:
# LC 455 — Assign Cookies
# Match smallest sufficient cookie to each child greedily
def findContentChildren(g, s):
    g.sort(); s.sort()
    child = cookie = 0
    while child < len(g) and cookie < len(s):
        if s[cookie] >= g[child]:       # cookie satisfies child
            child += 1
        cookie += 1                     # always move to next cookie
    return child


# LC 881 — Boats to Save People
# Pair heaviest with lightest if possible — two pointers after sorting
def numRescueBoats(people, limit):
    people.sort()
    left, right = 0, len(people) - 1
    boats = 0
    while left <= right:
        if people[left] + people[right] <= limit:
            left += 1                   # lightest can share with heaviest
        right -= 1                      # heaviest always gets a boat
        boats += 1
    return boats

## Common mistakes

- Sorting only one array when both need sorting — matching requires both to be ordered
- Moving both pointers when only one should move — in boats, only move `left` if paired; always move `right`
- Forgetting to increment `cookie` even when a child is not satisfied — must try the next cookie regardless

**Practice problems:**

| Problem | Sort strategy | Notes |
| :--- | :--- | :--- |
| LC 455 — Assign Cookies | Both ASC | Two pointers, move cookie always |
| LC 881 — Boats to Save People | ASC | Opposite ends two pointers |
| LC 870 — Advantage Shuffle | Both ASC | Match smallest winning card to each opponent card |

---
# Part 5 — Prefix / Running Choice

## Core idea

Scan left to right, making a greedy decision at each position based on accumulated information so far. No sorting needed — the order is fixed. The greedy choice is usually "can I afford to do this here, or should I reset?"

In [ ]:
# LC 134 — Gas Station
# Can we complete the circuit? If yes, find starting index.
# Greedy: if total gas >= total cost, a solution exists.
# Start from the point after the last deficit.
def canCompleteCircuit(gas, cost):
    total = tank = start = 0
    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total += diff
        tank  += diff
        if tank < 0:                    # can't reach next station from current start
            start = i + 1              # try starting from next station
            tank = 0
    return start if total >= 0 else -1  # total < 0 means no solution exists


# LC 135 — Candy
# Each child must have more candy than neighbors with lower rating.
# Two-pass greedy: left-to-right then right-to-left.
def candy(ratings):
    n = len(ratings)
    candies = [1] * n
    for i in range(1, n):              # left pass: satisfy left neighbor condition
        if ratings[i] > ratings[i-1]:
            candies[i] = candies[i-1] + 1
    for i in range(n-2, -1, -1):      # right pass: satisfy right neighbor condition
        if ratings[i] > ratings[i+1]:
            candies[i] = max(candies[i], candies[i+1] + 1)
    return sum(candies)

## Common mistakes

- In gas station: checking only one direction — the total gas check proves existence; the running tank finds the start
- In candy: doing only one pass — one pass can't satisfy both left and right neighbor constraints simultaneously
- In two-pass problems: forgetting to take `max` in the second pass — the second pass should strengthen, not overwrite, the first pass result

**Practice problems:**

| Problem | Passes | Notes |
| :--- | :--- | :--- |
| LC 134 — Gas Station | One pass | Reset start after deficit; check total at end |
| LC 135 — Candy | Two passes | Left → right then right → left; take max |
| LC 406 — Queue Reconstruction by Height | Sort + insert | Sort DESC by height, insert by index |

---
# Decision Guide

```
Is the problem about intervals?
├── Select max non-overlapping        → Sort by END, keep earliest-ending   (LC 435)
├── Merge overlapping                 → Sort by START, extend if overlap     (LC 56)
└── Minimum coverage / arrows         → Sort by END, shoot at current end    (LC 452)

Is the problem about reachability on an array?
└── Jump Game pattern
    ├── Can you reach end?            → Track max_reach                      (LC 55)
    └── Minimum jumps?                → Boundary + farthest pattern          (LC 45)

Does the greedy choice involve frequency?
└── Use a max-heap
    ├── Task cooldown                 → Heap + cooldown queue                (LC 621)
    └── No adjacent duplicates        → Always place most frequent           (LC 767)

Does the problem involve matching two groups?
└── Sort + two pointers
    ├── Match smallest to smallest    → Both ASC, advance both when matched  (LC 455)
    └── Pair lightest with heaviest   → ASC, opposite ends two pointers      (LC 881)

Does the problem require scanning with a running decision?
├── Single pass                       → Reset on failure, check total        (LC 134)
└── Two passes                        → Left → right then right → left       (LC 135)

Should I use Greedy or DP?
├── Need ALL solutions                → Backtracking
├── Subproblems overlap               → DP
├── Greedy choice property holds AND  → Greedy
│   optimal substructure holds
└── Unsure                            → Try greedy, construct a counterexample;
                                        if found → use DP
```